In [20]:
import numpy as np
import tensorflow
from keras.models import Model
from keras import layers

In [3]:
!kaggle datasets download -d preetviradiya/english-hindi-dataset

Dataset URL: https://www.kaggle.com/datasets/preetviradiya/english-hindi-dataset
License(s): GNU Free Documentation License 1.3
100% 13.9M/13.9M [00:01<00:00, 11.4MB/s]



In [4]:
!unzip english-hindi-dataset.zip

Archive:  english-hindi-dataset.zip
  inflating: Dataset_English_Hindi.csv  


In [5]:
import pandas as pd
data = pd.read_csv("/content/Dataset_English_Hindi.csv")

In [6]:
data.head()

,English,Hindi
0,Help!,बचाओ!
1,Jump.,उछलो.
2,Jump.,कूदो.
3,Jump.,छलांग.
4,Hello!,नमस्ते।


In [9]:
import re

def clean_english(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^a-zA-Z?.!,]+", " ", text)
    return text.strip()

def clean_hindi(text):
    text = str(text).strip()
    # Keep Hindi Devanagari characters (\u0900-\u097F) and basic punctuation
    text = re.sub(r"[^\u0900-\u097F?.!,]+", " ", text)
    return f"[start] {text.strip()} [end]"

eng_texts = data['English'].apply(clean_english).tolist()
hin_texts = data['Hindi'].apply(clean_hindi).tolist()

In [10]:
MAX_VOCAB = 15000
MAX_LEN = 30

eng_vectorizer = layers.TextVectorization(
    max_tokens=MAX_VOCAB, output_sequence_length=MAX_LEN
)
hin_vectorizer = layers.TextVectorization(
    max_tokens=MAX_VOCAB,
    output_sequence_length=MAX_LEN + 1,
    standardize=None
)

eng_vectorizer.adapt(eng_texts)
hin_vectorizer.adapt(hin_texts)

In [11]:
encoder_input_data = eng_vectorizer(eng_texts)
hin_sequences = hin_vectorizer(hin_texts)

decoder_input_data = hin_sequences[:, :-1]   # Everything except [end]
decoder_target_data = hin_sequences[:, 1:]   # Everything except [start]

In [14]:
EMB_DIM = 256
LATENT_DIM = 512

eng_vocab_size = len(eng_vectorizer.get_vocabulary())
hin_vocab_size = len(hin_vectorizer.get_vocabulary())

# --- ENCODER ---
encoder_inputs = layers.Input(shape=(None,), name="encoder_inputs")
# Encoder Embedding
enc_emb = layers.Embedding(eng_vocab_size, EMB_DIM, mask_zero=False)(encoder_inputs)

# LSTM
encoder_lstm = layers.LSTM(LATENT_DIM, return_state=True, name="encoder_lstm")
_, state_h, state_c = encoder_lstm(enc_emb)

# Context Vector
encoder_states = [state_h, state_c]

# --- DECODER ---
decoder_inputs = layers.Input(shape=(None,), name="decoder_inputs")

# Decoder Embedding
dec_emb_layer = layers.Embedding(hin_vocab_size, EMB_DIM, mask_zero=False)
dec_emb = dec_emb_layer(decoder_inputs)

# LSTM
decoder_lstm = layers.LSTM(LATENT_DIM, return_sequences=True, return_state=True, name="decoder_lstm")
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

# Softmax Layer
decoder_dense = layers.Dense(hin_vocab_size, activation="softmax", name="decoder_dense")
decoder_outputs = decoder_dense(decoder_outputs)

# --- FULL TRAINING MODEL ---
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, None, 256) │  3,840,000 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, None, 256) │  3,840,000 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 512),     │  1,574,912 │ embedding_2[0][0] │
│                     │ (None, 512),      │            │                   │
│                     │ (None, 512)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, None,     │  1,574,912 │ embedding_3[0][0… │
│                     │ 512), (None,      │            │ encoder_lstm[0][… │
│                     │ 512), (None,      │            │ encoder_lstm[0][… │
│                     │ 512)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, None,      │  7,695,000 │ decoder_lstm[0][… │
│ (Dense)             │ 15000)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 18,524,824 (70.67 MB)

 Trainable params: 18,524,824 (70.67 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
BATCH_SIZE = 64
EPOCHS = 5

history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.2
)

Epoch 1/5
1631/1631 ━━━━━━━━━━━━━━━━━━━━ 215s 132ms/step - accuracy: 0.7691 - loss: 1.0531 - val_accuracy: 0.6627 - val_loss: 2.0437
Epoch 2/5
1631/1631 ━━━━━━━━━━━━━━━━━━━━ 215s 132ms/step - accuracy: 0.7851 - loss: 0.9691 - val_accuracy: 0.6631 - val_loss: 2.0827
Epoch 3/5
1631/1631 ━━━━━━━━━━━━━━━━━━━━ 213s 131ms/step - accuracy: 0.7993 - loss: 0.8970 - val_accuracy: 0.6634 - val_loss: 2.1179
Epoch 4/5
1631/1631 ━━━━━━━━━━━━━━━━━━━━ 214s 131ms/step - accuracy: 0.8123 - loss: 0.8321 - val_accuracy: 0.6632 - val_loss: 2.1626
Epoch 5/5
1631/1631 ━━━━━━━━━━━━━━━━━━━━ 229s 140ms/step - accuracy: 0.8246 - loss: 0.7723 - val_accuracy: 0.6624 - val_loss: 2.2105


In [23]:
# 1. Standalone Encoder Model
encoder_model = Model(encoder_inputs, encoder_states)

# 2. Standalone Decoder Model
decoder_state_input_h = layers.Input(shape=(LATENT_DIM,))
decoder_state_input_c = layers.Input(shape=(LATENT_DIM,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

dec_emb_inf = dec_emb_layer(decoder_inputs)
decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    dec_emb_inf, initial_state=decoder_states_inputs
)
decoder_states_inf = [state_h_inf, state_c_inf]
decoder_outputs_inf = decoder_dense(decoder_outputs_inf)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs_inf] + decoder_states_inf
)

# Vocabulary lookup maps
hin_vocab = hin_vectorizer.get_vocabulary()
hin_word_to_idx = {word: idx for idx, word in enumerate(hin_vocab)}
hin_idx_to_word = {idx: word for idx, word in enumerate(hin_vocab)}

# 3. Translation Function
def translate_sentence(input_sentence):
    cleaned = clean_english(input_sentence)
    input_seq = eng_vectorizer([cleaned])
    states_value = encoder_model.predict(input_seq, verbose=0)

    # Start with the [start] token
    target_seq = np.array([[hin_word_to_idx["[start]"]]])
    decoded_words = []

    for _ in range(MAX_LEN):
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = hin_idx_to_word.get(sampled_token_index, "")

        if sampled_word == "[end]" or sampled_word == "":
            break

        decoded_words.append(sampled_word)

        # Update target sequence and states for the next word
        target_seq = np.array([[sampled_token_index]])
        states_value = [h, c]

    return " ".join(decoded_words)

# Example Usage:
print(translate_sentence("How are you?"))


आप कैसे [UNK]
